# LLM Arena: Gomoku (OpenAI vs Gemini)

这个 Notebook 构建了一个可扩展的 **LLM Arena** 基础版本：

- **Game Engine（Referee）**：维护 Gomoku 棋盘、合法落子、胜负判断
- **Base LLM Agent（Interface）**：统一玩家接口
- **API Adapters（Competitors）**：OpenAI 与 Gemini 适配器
- **Orchestrator（Arena）**：回合循环、无效输出重试、推理日志

> 设计目标：后续可平滑扩展到 5~6 个模型，仅需新增 Adapter 类。

In [2]:
# 如果缺依赖，请取消注释运行：
# !pip install -q -r requirements.txt

import os
import importlib
import pandas as pd

import arena_engine
import arena_agents
import arena_orchestrator
import arena_ui

# 关键：每次运行都重载本地模块，避免旧内核缓存导致参数签名不一致
importlib.reload(arena_engine)
importlib.reload(arena_agents)
importlib.reload(arena_orchestrator)
importlib.reload(arena_ui)

from arena_engine import GomokuEngine
from arena_agents import OpenAIAgent, GeminiAgent, RandomAgent
from arena_orchestrator import ArenaOrchestrator
from arena_ui import ArenaLiveWindow

## 1) API Key 配置区（请在这里填写）

你可以直接填写字符串，也可以使用环境变量。

In [3]:
# 方式 A：直接填写（推荐测试时使用）
OPENAI_API_KEY = ""    # 例如: sk-...
GEMINI_API_KEY = ""    # 例如: AIza...

# 方式 B：从环境变量读取（若上面为空则回退到环境变量）
if not OPENAI_API_KEY:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
if not GEMINI_API_KEY:
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

print("OpenAI key loaded:", bool(OPENAI_API_KEY))
print("Gemini key loaded:", bool(GEMINI_API_KEY))

OpenAI key loaded: True
Gemini key loaded: True


In [ ]:
# 核心类已迁移到同目录脚本文件，便于维护：
# - arena_engine.py
# - arena_agents.py
# - arena_orchestrator.py
# 本单元保留为空，避免重复定义。

# Agent 接口和工具函数已迁移至 arena_agents.py
# OpenAI / Gemini 适配器已迁移至 arena_agents.py
# Arena 主循环编排器已迁移至 arena_orchestrator.py

## 2) 组装对战（OpenAI vs Gemini）

当前 Notebook 只保留配置与主流程；核心逻辑已拆分到：

- `arena_engine.py`
- `arena_agents.py`
- `arena_orchestrator.py`
- `arena_ui.py`

对局开始前，系统会先向双方 LLM 发送同一份开局规则提示（含棋盘参数、胜利条件、坐标规则）。

当前开局规则启用：**SWAP（Pie Rule，职业常见交换先手规则）**

- 黑先落第一子
- 白随后选择：`[action] swap` 或 `[action] keep`
- 若白选择 swap，则双方执黑白互换

从正式回合开始：

- 每回合只反馈给当前 LLM：对手上一手信息（由 `hint` 参数控制）
- `hint=True`：发送 `action + reason`
- `hint=False`：只发送 `action`（不发送 reason）
- 不会在每回合重复提供全局棋盘状态
- LLM 需要依赖对话历史来重建局面（长记忆测试）

每一步要求 LLM 输出格式：

`[action] place at row X, column Y; [reason] (less than 100 words)`

合法性裁判规则：

- 每回合都会校验落子是否合法（包含重复落子、越界、解析失败、API调用失败等）
- 同一回合最多重试 3 次
- 超过 3 次仍不合法则该玩家直接判负（forfeit）

运行主单元后会先弹出一个窗口：

- 上方：棋盘可视化
- 下方：滚动日志（Agent->LLM 内容、LLM 原始输出、落子记录、swap 决策）
- 判定胜负后弹窗提示并结束游戏

每局结束会自动在 `arena_logs/<时间戳>/` 保存完整日志：

- `turn_logs.csv`
- `turn_logs.jsonl`
- `event_logs.jsonl`
- `summary.json`
- `final_board.json`

Gemini 适配器已迁移到 `google.genai`（新 SDK）。

如果某个 key 没填，自动用 `RandomAgent` 占位，方便先测试流程。

In [7]:
# 可按需调整棋盘大小；标准 Gomoku 常用 15x15，这里默认 9x9 便于快速测试
engine = GomokuEngine(size=15, win_len=5)

# hint 开关：
# True  -> 每回合给对方上一步 action + reason
# False -> 每回合只给对方上一步 action（不暴露 reason）
HINT = True

# 常用模型（示例，不是完整列表）：
# OpenAI: gpt-4o-mini（轻量快）, o3-mini（推理型）
# Gemini: gemini-2.5-flash（快）, gemini-2.5-pro（更强推理）
OPENAI_MODEL = "gpt-4o-mini"
GEMINI_MODEL = "gemini-2.5-pro"

if OPENAI_API_KEY:
    black = OpenAIAgent(api_key=OPENAI_API_KEY, marker=1, model=OPENAI_MODEL, name="OpenAI")
else:
    black = RandomAgent(name="OpenAI-Mock", marker=1)

if GEMINI_API_KEY:
    white = GeminiAgent(api_key=GEMINI_API_KEY, marker=-1, model=GEMINI_MODEL, name="Gemini")
else:
    white = RandomAgent(name="Gemini-Mock", marker=-1)

# 弹窗 UI：上方棋盘、下方日志（Agent->LLM 与 LLM 输出）
live_ui = ArenaLiveWindow(engine=engine, title="LLM Arena - OpenAI vs Gemini")

arena = ArenaOrchestrator(
    engine=engine,
    black_agent=black,
    white_agent=white,
    max_retries=2,
    render_each_turn=False,   # 使用 Tk 弹窗可视化，不使用 matplotlib 每步弹图
    event_callback=live_ui.handle_event,
    turn_delay_sec=0.4,
    opening_rule="swap",     # 专业交换先手规则（Pie Rule）
    illegal_retry_limit=3,    # 单回合非法动作最多重试 3 次，超限判负
    log_root_dir="arena_logs",  # 自动保存完整日志到子目录
    hint=HINT,
)

live_ui.run_with_orchestrator(arena, max_turns=120)

logs_df = pd.DataFrame(arena.logs)
print("Result:", arena.winner_text())
print("Hint mode:", HINT)
print("Models:", OPENAI_MODEL, "vs", GEMINI_MODEL)
print("Saved logs folder:", arena.output_dir)
logs_df.tail(10)

Result: BLACK wins
Hint mode: True
Models: gpt-4o-mini vs gemini-2.5-pro
Saved logs folder: arena_logs\20260222_205639


,turn_index,player,agent,model,row,col,applied,reasoning,raw_response,strict_format_matched,agent_message,opponent_last_action
8,6,WHITE,OpenAI,gpt-4o-mini,6.0,5.0,True,This move creates a potential horizontal line ...,"[place] place at row 6, column 5; [reason] Thi...",False,Opponent previous action: [action] place at ro...,"[action] place at row 8, column 6; [reason] Th..."
9,7,BLACK,Gemini,gemini-2.5-pro,6.0,7.0,True,This is a dual-purpose move. It blocks your de...,"[action] place at row 6, column 7; [reason] Th...",True,Opponent previous action: [action] place at ro...,"[action] place at row 6, column 5; [reason] Th..."
10,8,WHITE,OpenAI,gpt-4o-mini,5.0,6.0,True,This move blocks my opponent's vertical threat...,"[place] place at row 5, column 6; [reason] Thi...",False,Opponent previous action: [action] place at ro...,"[action] place at row 6, column 7; [reason] Th..."
11,9,BLACK,Gemini,gemini-2.5-pro,4.0,6.0,True,This is a critical defensive move. The opponen...,"[action] place at row 4, column 6; [reason] Th...",True,Opponent previous action: [action] place at ro...,"[action] place at row 5, column 6; [reason] Th..."
12,10,WHITE,OpenAI,gpt-4o-mini,5.0,5.0,True,This move creates a diagonal threat while also...,"[place] place at row 5, column 5; [reason] Thi...",False,Opponent previous action: [action] place at ro...,"[action] place at row 4, column 6; [reason] Th..."
13,11,BLACK,Gemini,gemini-2.5-pro,8.0,7.0,True,This move creates an open three-in-a-row for m...,"[action] place at row 8, column 7; [reason] Th...",True,Opponent previous action: [action] place at ro...,"[action] place at row 5, column 5; [reason] Th..."
14,12,WHITE,OpenAI,gpt-4o-mini,7.0,5.0,True,This move blocks my opponent's open three-in-a...,"[place] place at row 7, column 5; [reason] Thi...",False,Opponent previous action: [action] place at ro...,"[action] place at row 8, column 7; [reason] Th..."
15,13,BLACK,Gemini,gemini-2.5-pro,8.0,9.0,True,"The opponent's last move at (7,5) failed to bl...","[action] place at row 8, column 9; [reason] Th...",True,Opponent previous action: [action] place at ro...,"[action] place at row 7, column 5; [reason] Th..."
16,14,WHITE,OpenAI,gpt-4o-mini,8.0,10.0,True,This move blocks my opponent's open four-in-a-...,"[place] place at row 8, column 10; [reason] Th...",False,Opponent previous action: [action] place at ro...,"[action] place at row 8, column 9; [reason] Th..."
17,15,BLACK,Gemini,gemini-2.5-pro,8.0,5.0,True,My opponent only blocked one side of my four-i...,"[action] place at row 8, column 5; [reason] My...",True,Opponent previous action: [action] place at ro...,"[action] place at row 8, column 10; [reason] T..."


## 3) 分析日志

这里可以查看每一步模型选择与理由。

In [8]:
# 展示关键列
if len(logs_df) > 0:
    display(logs_df[["turn_index", "player", "agent", "model", "row", "col", "reasoning", "applied"]])
else:
    print("No logs yet. Run the arena cell first.")

,turn_index,player,agent,model,row,col,reasoning,applied
0,0,BLACK,OpenAI,gpt-4o-mini,NaN,NaN,Initial shared rules prompt sent.,True
1,0,WHITE,Gemini,gemini-2.5-pro,NaN,NaN,Initial shared rules prompt sent.,True
2,1,BLACK,OpenAI,gpt-4o-mini,7.0,7.0,This central position maximizes control over t...,True
3,1,WHITE,Gemini,gemini-2.5-pro,NaN,NaN,White chose to swap colors.,True
4,2,WHITE,OpenAI,gpt-4o-mini,7.0,6.0,This move creates a potential horizontal line ...,True
5,3,BLACK,Gemini,gemini-2.5-pro,8.0,8.0,This move creates a diagonal two-in-a-row conn...,True
6,4,WHITE,OpenAI,gpt-4o-mini,6.0,6.0,This move reinforces my position by creating a...,True
7,5,BLACK,Gemini,gemini-2.5-pro,8.0,6.0,This is a crucial defensive and offensive move...,True
8,6,WHITE,OpenAI,gpt-4o-mini,6.0,5.0,This move creates a potential horizontal line ...,True
9,7,BLACK,Gemini,gemini-2.5-pro,6.0,7.0,This is a dual-purpose move. It blocks your de...,True


## 4) 扩展到更多模型（Strategy + Adapter）

新增模型时只需：

1. 新建一个 `XXXAgent(BaseLLMAgent)`
2. 在 `generate_move()` 内把统一 prompt 映射到对应厂商 API
3. 在组装区把 `black/white` 之一替换成新 Agent

这样无需改 `GomokuEngine` 与 `ArenaOrchestrator` 主逻辑。